In [ ]:
import dataclasses, time
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
from collections import namedtuple
import jax
import jax.numpy as jnp
import jax.scipy.linalg
from itertools import combinations

jax.config.update("jax_enable_x64", True)
from pyblock3.fcidump import FCIDUMP
from pyblock3.hamiltonian import Hamiltonian
from pyblock3.algebra.mpe import MPE
from pyblock3.algebra.symmetry import SZ

from trot.core.system import System
from trot.core.ops import k_energy
from trot.ham.hubbard import HamHubbard
from trot.ham.chol import HamChol
from trot.trial.uhf import UhfTrial, get_rdm1 as uhf_get_rdm1
from trot.trial.auto import make_auto_trial_ops
from trot.core.ops import MeasOps
from trot.meas.uhf import energy_kernel_uw_rh, build_meas_ctx as uhf_build_meas_ctx
from trot.prop import blocks
from trot.prop.cpmc import init_prop_state
from trot.prop.hubbard_cpmc_ops import _build_prop_ctx, make_hubbard_cpmc_ops
from trot.prop.types import PropOps, PropState
from trot import walkers as wk
from trot.walkers import _qr as _qr_t
from trot.prop.types import QmcParams
from trot.driver import run_qmc_energy

In [35]:
L = 16
n_up = 4
n_down = 4 
t = 1.0
U = 4.0

h1 = np.zeros((L, L))
for i in range(L - 1):
    h1[i, i + 1] = h1[i + 1, i] = -t

ham = HamHubbard(h1=jnp.asarray(h1), u=U)
sys_ = System(norb=L, nelec=(n_up, n_down), walker_kind="unrestricted")

eps, mo = np.linalg.eigh(h1)
Ca, Cb = mo[:, :n_up].copy(), mo[:, :n_down].copy()
e_hf = 2.0 * eps[:n_up].sum() + U * sum((Ca[i] @ Ca[i]) * (Cb[i] @ Cb[i]) for i in range(L))

# walkers are initialised from this
trial_data = UhfTrial(mo_coeff_a=jnp.asarray(Ca), mo_coeff_b=jnp.asarray(Cb))
def energy_uhf(walker, ham_data=None, meas_ctx=None, trial_data=trial_data):
    """Single-determinant Hubbard local energy, used only as a cross-check of the
    MPS/MSD estimators below."""
    return energy_kernel_uw_rh(walker, ham_chol, meas_ctx_uhf, trial_data)
# the Hubbard interaction is exactly Cholesky-decomposable -- g2e[i,i,i,i] = U gives
# one vector per site, L^g = sqrt(U) E_gg -- which is what meas/uhf's kernel wants.
chol = np.zeros((L, L, L))
for i in range(L):
    chol[i, i, i] = np.sqrt(U)
ham_chol = HamChol(h0=jnp.asarray(0.0), h1=jnp.asarray(h1),
                   chol=jnp.asarray(chol), basis="restricted")
meas_ctx_uhf = uhf_build_meas_ctx(ham_chol, trial_data)

print(f"E_HF             = {e_hf:.12f}")

E_HF             = -9.783391410507


In [36]:
#Full ED benchmark, and the determinant basis the MSD route runs in. Est runtime 20s

# One string list serves both spins: the MSD kernels index alpha and beta with the
# same ROWS, which is only right at n_up == n_down.
assert n_up == n_down, "the MSD bookkeeping below indexes both spins with one string list"

STR = [frozenset(c) for c in combinations(range(L), n_up)]
IDX = {s: k for k, s in enumerate(STR)}
NS = len(STR)
OCCA = np.array([[1 if i in s else 0 for i in range(L)] for s in STR])
ROWS = [np.array(sorted(s)) for s in STR]


def _hop_matrix():
    """Spinless nearest-neighbour hopping in the string basis.

    The Jordan-Wigner string between ADJACENT sites is empty, so every nonzero
    element is just -t and there is no sign to track.
    """
    H = np.zeros((NS, NS))
    for k, s in enumerate(STR):
        for i in range(L - 1):
            for src, dst in ((i + 1, i), (i, i + 1)):
                if src in s and dst not in s:
                    H[IDX[frozenset((s - {src}) | {dst})], k] += -t
    return H


HOP = _hop_matrix()
DOCC = np.array([[len(a & b) for b in STR] for a in STR], float)


def apply_H(X):
    """H X in the determinant basis, X[a, b] indexed by (alpha string, beta string).

    H = sum_sigma sum_ij h1_ij c+_i c_j + U sum_i n_ia n_ib is one-body within each
    spin channel plus a diagonal, so applying it is two NS x NS matrix products and
    an elementwise scaling:

        (H X)[a,b] = sum_a' HOP[a,a'] X[a',b] + sum_b' HOP[b,b'] X[a,b'] + U d_ab X[a,b]

    No two-electron integrals and no FCI library. This is the MSD route's
    Hamiltonian; it shares no code with the MPO the MPS route uses, which is what
    makes comparing the two routes worth anything.
    """
    return HOP @ X + X @ HOP.T + U * DOCC * X


def ed_energy():
    """Ground state by dense eigh in the (n_up, n_down) sector."""
    H = np.zeros((NS * NS, NS * NS))
    Hr = H.reshape(NS, NS, NS, NS)          # a view, so these write into H
    for ib in range(NS):
        Hr[:, ib, :, ib] += HOP             # hopping, spin up
    for ia in range(NS):
        Hr[ia, :, ia, :] += HOP             # hopping, spin down
    H[np.diag_indices_from(H)] += U * DOCC.ravel()
    return float(np.linalg.eigvalsh(H)[0])


e_exact = ed_energy()          # ~20 s; to skip it set e_exact = -4.235806999129656
print(f"E_exact           = {e_exact:.12f}")

E_exact           = -4.235806999130


In [37]:
def plan_channel_maxB(C):
    """Fishman-White plan with B = n-k always: the remaining block is a genuine
    projector, so its eigenvalues are exactly 0/1 for any walker."""
    U, n = np.asarray(C, float).copy(), C.shape[0]
    occ, Bs, vrefs = np.zeros(n, int), [], []
    for k in range(n - 1):
        B = n - k
        w, W = np.linalg.eigh((U @ U.T)[k : k + B, k : k + B])
        v, occ[k] = (W[:, 0], 0) if w[0] <= 1.0 - w[-1] else (W[:, -1], 1)
        Bs.append(B); vrefs.append(v.copy())
        for j in range(B - 1, 0, -1):
            th = np.arctan2(v[j], v[j - 1])
            c, s = np.cos(th), np.sin(th)
            v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
            U[k + j - 1], U[k + j] = (c * U[k + j - 1] + s * U[k + j],
                                      -s * U[k + j - 1] + c * U[k + j])
    occ[n - 1] = int(round(float(U[n - 1] @ U[n - 1])))
    assert occ.sum() == C.shape[1], "particle number lost during compression"
    return occ, np.array(Bs), vrefs


def plan_channel(C, eps_occ=1e-10):
    """Fishman-White plan with B grown only until a mode isolates.

    The block is enlarged until some eigenvalue of Lambda_B comes within eps_occ
    of 0 or 1. For a low-entangled state that happens at small B, so both the
    gate count and the bond dimension stay far below the maximal-B plan -- which
    is what makes L > 16 reachable at all, since maximal B gives chi = 2^(L/2)
    (1024 at L = 20, 4096 at L = 24).

    The price: the retained block is only APPROXIMATELY idempotent, so the
    replay must use mode="eigh". See `channel_angles`.
    """
    U, n = np.asarray(C, float).copy(), C.shape[0]
    occ, Bs, vrefs = np.zeros(n, int), [], []
    for k in range(n - 1):
        Lam = U @ U.T
        for B in range(2, n - k + 1):                  # grow until a mode isolates
            w, W = np.linalg.eigh(Lam[k : k + B, k : k + B])
            if min(w[0], 1.0 - w[-1]) < eps_occ:
                break
        v, occ[k] = (W[:, 0], 0) if w[0] <= 1.0 - w[-1] else (W[:, -1], 1)
        Bs.append(B); vrefs.append(v.copy())
        for j in range(B - 1, 0, -1):
            th = np.arctan2(v[j], v[j - 1])
            c, s = np.cos(th), np.sin(th)
            v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
            U[k + j - 1], U[k + j] = (c * U[k + j - 1] + s * U[k + j],
                                      -s * U[k + j - 1] + c * U[k + j])
    occ[n - 1] = int(round(float(U[n - 1] @ U[n - 1])))
    assert occ.sum() == C.shape[1], "particle number lost during compression"
    return occ, np.array(Bs), vrefs


def is_max_b(plan, n):
    """True if the plan always takes the whole remaining block."""
    return all(int(B) == n - k for k, B in enumerate(plan[1]))


def channel_angles(C, plan, mode="auto", n_mv=2, xp=jnp):
    """Replay the Givens angles from C with the plan frozen. C must be ORTHONORMAL.

    mode="proj" -- MAXIMAL B ONLY. Lambda = C C^T is a projector and, at maximal
        B, so is the remaining block, so P = M (occupied) / I - M (empty) IS the
        spectral filter. `n_mv` matvecs against the plan's reference vector pick
        the mode out of the degenerate cluster and fix its sign; n_mv=2 rather
        than 1 because one projection of vref can be short (||P vref|| ~ 1e-2 for
        a drifted walker) and normalising it inflates roundoff, while a second
        projection has ||P v|| ~ 1 and cleans it up.

    mode="eigh" -- any B, and REQUIRED for a non-maximal plan: there the retained
        block is only approximately idempotent, P is not a projector, and two
        power-iteration steps do not converge on the mode. Measured on
        `plan_channel` plans, overlap error against the exact det(C_T^T C_w):

            L        proj      eigh
             8    6.0e-03   6.1e-16
            12    5.0e-02   4.2e-15
            16    4.9e-01   5.1e-04

        Nothing asserts on it -- the wrong filter just returns wrong overlaps --
        which is why mode="auto" reads the choice off the plan rather than
        leaving it to the caller.

    mode="auto" takes proj at maximal B (the faster of the two there) and eigh
    otherwise.
    """
    occ, Bs, vrefs = plan
    if mode == "auto":
        mode = "proj" if is_max_b(plan, len(occ)) else "eigh"
    C = xp.asarray(C)
    rows = [C[i] for i in range(C.shape[0])]
    angles = []
    for k, (B, vref) in enumerate(zip(Bs, vrefs)):
        B = int(B)
        Uk = xp.stack(rows[k : k + B])
        M = Uk @ Uk.T
        vr = xp.asarray(vref)
        if mode == "eigh":
            _, W = xp.linalg.eigh(M)
            v = W[:, -1] if occ[k] == 1 else W[:, 0]
        else:
            P = M if occ[k] == 1 else xp.eye(B) - M
            v = vr
            for _ in range(n_mv):
                v = P @ v
                v = v / xp.linalg.norm(v)
        v = v * xp.sign(v @ vr)
        v = [v[i] for i in range(B)]
        for j in range(B - 1, 0, -1):
            th = xp.arctan2(v[j], v[j - 1])
            c, s = xp.cos(th), xp.sin(th)
            v[j - 1] = c * v[j - 1] + s * v[j]
            p = k + j - 1
            rows[p], rows[p + 1] = (c * rows[p] + s * rows[p + 1],
                                    -s * rows[p] + c * rows[p + 1])
            angles.append((p, th))
    return angles, rows


def V_hat(th):
    """The two-site gate, as a (2,2,2,2) tensor. The reference definition;
    `_gate_pair` fuses it into the contraction."""
    c, s = jnp.cos(th), jnp.sin(th)
    g = jnp.eye(4).at[1, 1].set(c).at[1, 2].set(s).at[2, 1].set(-s).at[2, 2].set(c)
    return g.reshape(2, 2, 2, 2)


def _gate_pair(A, B, th, xp=jnp):
    """(A B) with the gate folded in, as (Dl, 2, 2, Dr)."""
    c, s = xp.cos(th), xp.sin(th)
    t00 = A[:, 0, :] @ B[:, 0, :]
    t01 = A[:, 0, :] @ B[:, 1, :]
    t10 = A[:, 1, :] @ B[:, 0, :]
    t11 = A[:, 1, :] @ B[:, 1, :]
    return xp.stack([xp.stack([t00, c * t01 + s * t10], 1),
                     xp.stack([-s * t01 + c * t10, t11], 1)], 1)


def sector_plan(ql, qr):
    """Static per-charge row/column index sets for one two-site split, plus the
    middle labels and the gather maps back to full shape. NumPy, hence cached."""
    nl, nr = len(ql), len(qr)
    rc = (ql[:, None] + np.arange(2)[None, :]).ravel()
    cc = (qr[None, :] - np.arange(2)[:, None]).ravel()
    secs, qm, rcat, ccat = [], [], [], []
    for nm in sorted(set(rc.tolist()) & set(cc.tolist())):
        r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
        k = min(len(r), len(c))
        secs.append((r, c, k)); qm += [nm] * k
        rcat.append(r); ccat.append(c)
    rcat, ccat = np.concatenate(rcat), np.concatenate(ccat)
    rmap = np.full(2 * nl, len(rcat), int); rmap[rcat] = np.arange(len(rcat))
    cmap = np.full(2 * nr, len(ccat), int); cmap[ccat] = np.arange(len(ccat))
    return secs, np.array(qm, int), rmap, cmap


_SEC_CACHE = {}

def _sectors(ql, qr):
    key = (ql.tobytes(), len(ql), qr.tobytes(), len(qr))
    if key not in _SEC_CACHE:
        _SEC_CACHE[key] = sector_plan(ql, qr)
    return _SEC_CACHE[key]


def split_full(T, ql, qr):
    """Exact split, full rank in each particle-number sector, via QR."""
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl * 2, 2 * Dr)
    secs, qm, rmap, cmap = _sectors(ql, qr)
    As, Bs = [], []
    for r, c, k in secs:
        q, rr = jnp.linalg.qr(M[np.ix_(r, c)], mode="reduced")
        As.append(q[:, :k]); Bs.append(rr[:k])
    A = jax.scipy.linalg.block_diag(*As)
    B = jax.scipy.linalg.block_diag(*Bs)
    A = jnp.concatenate([A, jnp.zeros((1, A.shape[1]), A.dtype)], 0)[rmap]
    B = jnp.concatenate([B, jnp.zeros((B.shape[0], 1), B.dtype)], 1)[:, cmap]
    return A.reshape(Dl, 2, -1), B.reshape(-1, 2, Dr), qm




def channel_mps(C, plan):
    """Product state |occ> -> gates in reverse derivation order -> one d=2 MPS.

    Returns (tensors, bond labels, gauge sign). This notebook fixes the gauge by
    the amplitude ratio in Part 3 instead, so the third value is unused here.
    """
    one_hot = (jnp.array([[[1.0], [0.0]]]), jnp.array([[[0.0], [1.0]]]))

    occ = plan[0]
    ts = [one_hot[int(o)] for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))
    angles, rows = channel_angles(C, plan)
    for p, th in reversed(angles):
        ts[p], ts[p + 1], qn[p + 1] = split_full(_gate_pair(ts[p], ts[p + 1], th),
                                                 qn[p], qn[p + 2])
    gauge = jnp.linalg.det(jnp.stack([rows[i] for i in np.where(occ == 1)[0]]))
    return ts, qn, gauge


def combine(Aa, qna, Ab, qnb):
    """Interleave two d=2 channels into one d=4 MPS, local index n_a + 2 n_b."""
    ts, qn = [], [np.zeros((1, 2), int)]
    for i in range(len(Aa)):
        Dal, _, Dar = Aa[i].shape
        Dbl, _, Dbr = Ab[i].shape
        out = jnp.zeros((Dal, Dbl, 4, Dar, Dbr))
        for na in (0, 1):
            sgn = (-1.0) ** (na * qnb[i])                 # the Jordan-Wigner sign
            for nb in (0, 1):
                out = out.at[:, :, na + 2 * nb, :, :].set(
                    jnp.einsum("ar,b,bs->abrs", Aa[i][:, na, :], sgn, Ab[i][:, nb, :]))
        ts.append(out.reshape(Dal * Dbl, 4, Dar * Dbr))
        qn.append(np.stack([np.repeat(qna[i + 1], Dbr), np.tile(qnb[i + 1], Dar)], 1))
    return ts, qn


def sd_to_mps_qn(ca, cb, plan_a, plan_b):
    """Slater determinant -> (d=4 MPS, per-bond (n_a, n_b) labels). ORTHONORMAL ca/cb."""
    ta, qna, _ = channel_mps(ca, plan_a)
    tb, qnb, _ = channel_mps(cb, plan_b)
    return combine(ta, qna, tb, qnb)


def sd_to_mps(ca, cb, plan_a, plan_b):

    return sd_to_mps_qn(ca, cb, plan_a, plan_b)[0]


def mps_overlap(bra, ket):
    """<bra|ket>, dense. The charge-blocked version is in Part 3."""
    e = jnp.ones((bra[0].shape[0], ket[0].shape[0]))
    for a, b in zip(bra, ket):
        e = jnp.tensordot(a, jnp.tensordot(e, b, ([1], [0])), ([0, 1], [0, 1]))
    return e.reshape(())


def hubbard_mpo(L, t, U):
    """Dw=6 MPO. Bond basis: 0 = nothing started, 1..4 = a hop is pending, 5 = done."""
    I4 = np.eye(4)
    cr_a = np.zeros((4, 4)); cr_a[1, 0] = 1.0; cr_a[3, 2] = 1.0
    cr_b = np.zeros((4, 4)); cr_b[2, 0] = 1.0; cr_b[3, 1] = -1.0
    an_a, an_b = cr_a.T.copy(), cr_b.T.copy()
    n_a, n_b = np.diag([0., 1., 0., 1.]), np.diag([0., 0., 1., 1.])
    P_a, P_b = np.diag([1., -1., 1., -1.]), np.diag([1., 1., -1., -1.])
    W = np.zeros((L, 6, 4, 4, 6))
    for i in range(L):
        W[i, 0, :, :, 0] = I4
        W[i, 5, :, :, 5] = I4
        W[i, 0, :, :, 5] = U * (n_a @ n_b)
        if i < L - 1:
            W[i, 0, :, :, 1] = cr_a @ P_b
            W[i, 0, :, :, 2] = an_a @ P_b
            W[i, 0, :, :, 3] = P_a @ cr_b
            W[i, 0, :, :, 4] = P_a @ an_b
        if i > 0:
            W[i, 1, :, :, 5] = -t * an_a
            W[i, 2, :, :, 5] = -t * cr_a
            W[i, 3, :, :, 5] = -t * an_b
            W[i, 4, :, :, 5] = -t * cr_b
    return W


def apply_mpo(W, ts):
    """(W psi)[i] has bond dimension Dw*chi; the boundary MPO bonds are projected out."""
    out = []
    for i, (w, A) in enumerate(zip(W, ts)):
        A = np.asarray(A)
        if i == 0:
            w = w[0:1]
        if i == len(ts) - 1:
            w = w[:, :, :, 5:6]
        T = np.einsum("apqb,cqd->acpbd", w, A)
        dl, cl, p, dr, cr = T.shape
        out.append(T.reshape(dl * cl, p, dr * cr))
    return out


def compress(ts, tol=1e-13):
    """Left-to-right QR to canonicalise, then right-to-left SVD. Exact at this tol."""
    ts = [np.asarray(t) for t in ts]
    for i in range(len(ts) - 1):
        Dl, d, Dr = ts[i].shape
        q, r = np.linalg.qr(ts[i].reshape(Dl * d, Dr))
        ts[i] = q.reshape(Dl, d, -1)
        ts[i + 1] = np.tensordot(r, ts[i + 1], axes=([1], [0]))
    for i in range(len(ts) - 1, 0, -1):
        Dl, d, Dr = ts[i].shape
        u, sv, vt = np.linalg.svd(ts[i].reshape(Dl, d * Dr), full_matrices=False)
        k = max(int((sv > tol * max(sv[0], 1e-300)).sum()), 1)
        ts[i] = vt[:k].reshape(k, d, Dr)
        ts[i - 1] = np.tensordot(ts[i - 1], u[:, :k] * sv[:k], axes=([2], [0]))
    return ts

In [39]:
def build_hamil(L, U, t=1.0):
    h1e = np.zeros((L, L))
    for i in range(L - 1):
        h1e[i, i + 1] = h1e[i + 1, i] = -t
    g2e = np.zeros((L,) * 4)
    for i in range(L):
        g2e[i, i, i, i] = U
    fd = FCIDUMP(pg="c1", n_sites=L, n_elec=n_up + n_down, twos=n_up - n_down,
                 ipg=0, h1e=h1e, g2e=g2e)
    return Hamiltonian(fd, flat=True)


def run_dmrg(hamil, bdim, n_sweeps=14, seed=0):
    
    np.random.seed(seed)    
    mpo, _ = hamil.build_qc_mpo().compress(cutoff=1e-12)
    mps = hamil.build_mps(bdim)
    dmrg = MPE(mps, mpo, mps).dmrg(bdims=[bdim] * n_sweeps, noises=[1e-5] * 6 + [0],
                                   dav_thrds=[1e-10], iprint=-1, n_sweeps=n_sweeps)
    return mps, float(dmrg.energies[-1])


def spin_occ(q):
    """(n_alpha, n_beta) of an SZ label: n = na + nb, 2Sz = na - nb."""
    return (int(q.n) + int(q.twos)) // 2, (int(q.n) - int(q.twos)) // 2


def flat_blocks(mps, i):
    """(q_labels, shape, data) for every block of site i, with Python-int labels."""
    t = mps[i]
    for k in range(t.n_blocks):
        q = tuple(SZ.from_flat(int(x)) for x in t.q_labels[k])
        sh = tuple(int(x) for x in t.shapes[k])
        yield q, sh, np.asarray(t.data[t.idxs[k]:t.idxs[k + 1]]).reshape(sh)


def densify(mps, L):
    """flat pyblock3 MPS -> dense (Dl, 4, Dr) arrays, local index n_a + 2 n_b."""
    qkey = lambda q: (int(q.n), int(q.twos))
    left, right = [], []
    for i in range(L):
        lo, ro = {}, {}
        for (ql, qp, qr), sh, _ in flat_blocks(mps, i):
            lo[qkey(ql)], ro[qkey(qr)] = sh[0], sh[2]
        left.append(lo); right.append(ro)
    for i in range(L - 1):
        assert left[i + 1] == right[i], f"bond {i+1} mismatch between sites"

    offs = []
    for b in [left[i] for i in range(L)] + [right[L - 1]]:
        o, acc = {}, 0
        for k in sorted(b):
            o[k] = (acc, b[k]); acc += b[k]
        offs.append((o, acc))

    out = []
    for i in range(L):
        A = np.zeros((offs[i][1], 4, offs[i + 1][1]))
        for (ql, qp, qr), sh, dat in flat_blocks(mps, i):
            assert sh[1] == 1, f"physical block dim {sh[1]} != 1"
            na, nb = spin_occ(qp)
            ol, dl = offs[i][0][qkey(ql)]
            orr, dr = offs[i + 1][0][qkey(qr)]
            A[ol:ol + dl, na + 2 * nb, orr:orr + dr] = dat[:, 0, :]
        out.append(A)
    return out


def bond_qns(mps, L):
    """Per-bond (n_a, n_b) label for every index of the densified MPS.
    The same offset bookkeeping `densify` uses, so the labels and the dense
    arrays are guaranteed to agree index for index. This is what the
    charge-blocked contraction in Part 3 needs.
    """
    qkey = lambda q: (int(q.n), int(q.twos))
    left, right = [], []
    for i in range(L):
        lo, ro = {}, {}
        for (ql, qp, qr), sh, _ in flat_blocks(mps, i):
            lo[qkey(ql)], ro[qkey(qr)] = sh[0], sh[2]
        left.append(lo); right.append(ro)
    out = []
    for b in [left[i] for i in range(L)] + [right[L - 1]]:
        lab = []
        for k in sorted(b):
            lab += [((k[0] + k[1]) // 2, (k[0] - k[1]) // 2)] * b[k]   # (n,2Sz)->(na,nb)
        out.append(np.array(lab, int))
    return out

def mps_amp(ts, occ_a, occ_b):
    """<d|MPS>, a bond-dimension-1 contraction."""
    v = np.ones((1, 1))
    for A, a, b in zip(ts, occ_a, occ_b):
        v = v @ np.asarray(A)[:, int(a) + 2 * int(b), :]
    return float(v[0, 0])


def interleaving_sign(ra, rb):
    """(-1)^K relating 'all alpha then all beta' to the interleaved lattice order."""
    return (-1.0) ** sum(int((rb < i).sum()) for i in ra)

CHI = 64                  
hamil = build_hamil(L, U)
mps_dmrg, E_dmrg_dav = run_dmrg(hamil, CHI)

LOCAL = {spin_occ(SZ.from_flat(int(c))): k for k, c in enumerate(hamil.basis[0])}
ket_T_np = densify(mps_dmrg, L)
ket_T = [jnp.asarray(t) for t in ket_T_np]

print(f"DMRG chi={CHI}:  bond dims {mps_dmrg.show_bond_dims()}")
print(f"local index map read from hamil.basis: {LOCAL}   -> l = n_a + 2 n_b")
print(f"dense bond dims : {[t.shape[0] for t in ket_T] + [ket_T[-1].shape[-1]]}")
print(f"<psi_T|psi_T>   = {float(mps_overlap(ket_T, ket_T)):.12f}")

# The trial in the determinant basis. STR / OCCA / ROWS / NS come from the ED cell.
ISG = np.array([[interleaving_sign(ROWS[a], ROWS[b]) for b in range(NS)] for a in range(NS)])
AMP = np.array([[mps_amp(ket_T_np, OCCA[a], OCCA[b]) for b in range(NS)] for a in range(NS)])
M = ISG * AMP
print(f"|c_d| > 1e-10 : {(np.abs(AMP) > 1e-10).sum()} of {NS * NS} determinants")

MH = apply_H(M)                                        # the MSD route's H|psi_T>
Hket_T = [jnp.asarray(t) for t in compress(apply_mpo(hubbard_mpo(L, t, U), ket_T_np))]

e_T = float((M * MH).sum() / (M * M).sum())
print(f"H|psi_T> bond dims : {[t.shape[0] for t in Hket_T] + [Hket_T[-1].shape[-1]]}")
print(f"<psi_T|H|psi_T>    = {e_T:.12f}")
print(f"E_exact            = {e_exact:.12f}")
print(f"trial error        = {e_T - e_exact:+.3e}")

QC MPO site   0 / 16
QC MPO site   1 / 16
QC MPO site   2 / 16
QC MPO site   3 / 16
QC MPO site   4 / 16
QC MPO site   5 / 16
QC MPO site   6 / 16
QC MPO site   7 / 16
QC MPO site   8 / 16
QC MPO site   9 / 16
QC MPO site  10 / 16
QC MPO site  11 / 16
QC MPO site  12 / 16
QC MPO site  13 / 16
QC MPO site  14 / 16
QC MPO site  15 / 16
DMRG chi=64:  bond dims 1|4|16|48|64|64|64|64|64|64|64|64|64|51|16|4|1
local index map read from hamil.basis: {(0, 0): 0, (1, 0): 1, (0, 1): 2, (1, 1): 3}   -> l = n_a + 2 n_b
dense bond dims : [1, 4, 16, 48, 64, 64, 64, 64, 64, 64, 64, 64, 64, 51, 16, 4, 1]
<psi_T|psi_T>   = 1.000000000000
|c_d| > 1e-10 : 3066958 of 3312400 determinants
H|psi_T> bond dims : [1, 4, 16, 59, 125, 196, 242, 234, 228, 238, 241, 196, 126, 61, 16, 4, 1]
<psi_T|H|psi_T>    = -11.686025135156
E_exact            = -4.235806999130
trial error        = -7.450e+00


In [44]:
Mj, MHj = jnp.asarray(M), jnp.asarray(MH)
RA = jnp.asarray(np.stack(ROWS))                      # (70, 4) occupied rows per string


def _dets(c):
    """det(C[r, :]) for all 70 strings."""
    return jax.vmap(lambda r: jnp.linalg.det(c[r, :]))(RA)


def msd_overlap(walker, trial_data=None):
    ca, cb = walker
    return _dets(ca) @ Mj @ _dets(cb)


def msd_energy(walker, ham_data=None, meas_ctx=None, trial_data=None):
    ca, cb = walker
    da, db = _dets(ca), _dets(cb)
    return (da @ MHj @ db) / (da @ Mj @ db)


# --- the MPS route: convert the walker, then contract. Plan is HF-based and fixed.
PLAN_B = "maximal"      # "maximal": exact for any orthonormal walker, chi = 2^(L/2)
                        # "adaptive": far smaller chi, but the frozen plan stops
                        #   transferring exactly (1e-15 at L<=12, 5e-4 at L=16)
_mk_plan = plan_channel_maxB if PLAN_B == "maximal" else plan_channel
plan_a, plan_b = _mk_plan(Ca), _mk_plan(Cb)
print(f"plan: {PLAN_B} B, {int((plan_a[1]-1).sum())} gates, "
      f"filter {'proj' if is_max_b(plan_a, L) else 'eigh'}")
ROWS_A, ROWS_B = np.where(plan_a[0] == 1)[0], np.where(plan_b[0] == 1)[0]
ISIGN_REF = interleaving_sign(ROWS_A, ROWS_B)
LOC_REF = [int(a) + 2 * int(b) for a, b in zip(plan_a[0], plan_b[0])]


def amp_exact_ref(ca, cb):
    return ISIGN_REF * jnp.linalg.det(ca[ROWS_A, :]) * jnp.linalg.det(cb[ROWS_B, :])


def amp_mps_ref(ts):
    v = jnp.ones((1, 1))
    for A, l in zip(ts, LOC_REF):
        v = v @ A[:, l, :]
    return v[0, 0]


def _bra(walker):
    ca, cb = walker
    qa, _ = jnp.linalg.qr(ca)
    qb, _ = jnp.linalg.qr(cb)
    return sd_to_mps(qa, qb, plan_a, plan_b), ca, cb


def mps_overlap_T(walker, trial_data=None):
    bra, ca, cb = _bra(walker)
    return (amp_exact_ref(ca, cb) / amp_mps_ref(bra)) * mps_overlap(bra, ket_T)


def mps_energy_T(walker, ham_data=None, meas_ctx=None, trial_data=None):
    bra, _, _ = _bra(walker)
    return mps_overlap(bra, Hket_T) / mps_overlap(bra, ket_T)

---
### The charge-blocked overlap

Both the walker MPS and the DMRG trial conserve $(n_\alpha, n_\beta)$, so the
environment is nonzero only between bond indices carrying the *same* charge.
Exploiting that with sparse kernels would not port to a GPU, so instead each
bond is sorted by charge and each sector padded to a rectangle: the environment
becomes one dense array `E[q, a, b]` and a site update becomes two **batched
GEMMs** plus a segment-sum, with every shape static. The padding is zero, so the
result is exact rather than an approximation.

In [45]:
"""The charge-blocked overlap: dense, batched, GPU-shaped.

Both MPSs conserve (n_alpha, n_beta), so the environment E[c, r] is nonzero only
where the bra bond index c and the ket bond index r carry the SAME charge. The
point is to use that WITHOUT sparse kernels, which do not port to a GPU: sort
every bond by charge, pad each charge sector to a common size, and carry the
environment as one dense array

    E[q, a, b]        q = charge, a = bra index within q, b = ket index within q

A site update is then two BATCHED GEMMs and one segment-sum, every shape static,
no data-dependent indexing:

    Ein[t] = E[src[t]]                           gather, static indices
    tmp[t] = bra_blk[t]^T Ein[t]                 batched GEMM
    out[t] = tmp[t] ket_blk[t]                   batched GEMM
    E'[q'] = sum_{t : dst[t] = q'} out[t]        segment-sum

`t` runs over the allowed (charge in, physical index, charge out) transitions,
fixed by the bond labels, so the whole layout is planned once in NumPy -- the
same plan/replay split used for the two-site splits.

Padding is zero and the padded entries of the blocks are zero, so this is EXACT:
it returns the same number as the dense contraction, not an approximation.

Two things make it pay here beyond the charge blocking itself:

  * only charges present on BOTH sides survive. A bra charge the ket does not
    have can never reach the last bond, so all work on it is dropped. With a
    chi=8 trial against a chi=256 walker that removes most of the walker.
  * the ket is FIXED, so its padded blocks are extracted once, in NumPy, and
    there is no runtime gather on that side.
"""
PHYS_NAB = np.array([[l % 2, l // 2] for l in range(4)])      # l -> (n_a, n_b)


def _charge_index(qn):
    """{(na,nb): array of bond indices} for one bond's label array."""
    out = {}
    for i, q in enumerate(map(tuple, np.asarray(qn).tolist())):
        out.setdefault(q, []).append(i)
    return {k: np.array(v, int) for k, v in out.items()}


def block_plan(qn_bra, qn_ket):
    """Static layout and transition table for every site. All NumPy."""
    n = len(qn_bra) - 1
    bidx = [_charge_index(q) for q in qn_bra]
    kidx = [_charge_index(q) for q in qn_ket]
    charges, A, B = [], [], []
    for x in range(n + 1):
        q = sorted(set(bidx[x]) & set(kidx[x]))
        charges.append(q)
        A.append(max([len(bidx[x][c]) for c in q], default=0))
        B.append(max([len(kidx[x][c]) for c in q], default=0))

    sites = []
    for x in range(n):
        pos_in = {c: i for i, c in enumerate(charges[x])}
        pos_out = {c: i for i, c in enumerate(charges[x + 1])}
        src, dst, ri, ci, mb, lid = [], [], [], [], [], []
        for c in charges[x]:
            rows = bidx[x][c]
            for l in range(4):
                c2 = (c[0] + PHYS_NAB[l][0], c[1] + PHYS_NAB[l][1])
                if c2 not in pos_out:
                    continue
                cols = bidx[x + 1][c2]
                R = np.zeros((A[x], A[x + 1]), int)      # padded index grids; the
                C = np.zeros((A[x], A[x + 1]), int)      # padding points at 0 and
                M = np.zeros((A[x], A[x + 1]))           # is masked to zero
                R[:len(rows), :len(cols)] = rows[:, None]
                C[:len(rows), :len(cols)] = cols[None, :]
                M[:len(rows), :len(cols)] = 1.0
                src.append(pos_in[c]); dst.append(pos_out[c2])
                ri.append(R); ci.append(C); mb.append(M); lid.append(l)
        sites.append(dict(src=np.array(src, int), dst=np.array(dst, int),
                          ri=np.stack(ri), ci=np.stack(ci), mb=np.stack(mb),
                          lid=np.array(lid, int), nq_out=len(charges[x + 1])))
    return dict(sites=sites, charges=charges, A=A, B=B, bidx=bidx, kidx=kidx, n=n)


def ket_blocks(ket, plan):
    """Pre-extract the fixed side's padded blocks once: no runtime gather there."""
    out = []
    for x, st in enumerate(plan["sites"]):
        blk = np.zeros((len(st["src"]), plan["B"][x], plan["B"][x + 1]))
        for t, (qi, qo, l) in enumerate(zip(st["src"], st["dst"], st["lid"])):
            c, c2 = plan["charges"][x][qi], plan["charges"][x + 1][qo]
            rows, cols = plan["kidx"][x][c], plan["kidx"][x + 1][c2]
            blk[t, :len(rows), :len(cols)] = \
                np.asarray(ket[x])[np.ix_(rows, [l], cols)][:, 0, :]
        out.append(jnp.asarray(blk))
    return out


def blocked_overlap(bra, kblk, plan):
    """<bra|ket> with the charge structure carried as dense padded blocks."""
    E = jnp.ones((1, plan["A"][0], plan["B"][0]))
    for x, (st, kb) in enumerate(zip(plan["sites"], kblk)):
        bb = bra[x][st["ri"], st["lid"][:, None, None], st["ci"]] * st["mb"]
        Ein = E[st["src"]]                                  # (T, A_x,   B_x)
        tmp = jnp.einsum("tij,tik->tjk", bb, Ein)           # (T, A_x+1, B_x)
        out = jnp.einsum("tjk,tkl->tjl", tmp, kb)           # (T, A_x+1, B_x+1)
        E = jax.ops.segment_sum(out, st["dst"], num_segments=st["nq_out"])
    return E.reshape(())


def plan_report(plan):
    """stored environment entries: padded blocks vs exact blocks vs dense."""
    pad = sum(len(plan["charges"][x]) * plan["A"][x] * plan["B"][x]
              for x in range(plan["n"] + 1))
    exact = sum(sum(len(plan["bidx"][x][c]) * len(plan["kidx"][x][c])
                    for c in plan["charges"][x]) for x in range(plan["n"] + 1))
    dense = sum(sum(len(v) for v in plan["bidx"][x].values())
                * sum(len(v) for v in plan["kidx"][x].values())
                for x in range(plan["n"] + 1))
    return dict(dense=dense, padded_blocks=pad, exact_blocks=exact,
                transitions=sum(len(st["src"]) for st in plan["sites"]))


# ------------------------------------------------- build the plan for this trial
qn_ket = bond_qns(mps_dmrg, L)
_bra_ref, qn_bra = sd_to_mps_qn(jnp.asarray(Ca), jnp.asarray(Cb), plan_a, plan_b)

blk_plan = block_plan(qn_bra, qn_ket)
ket_T_blk = ket_blocks(ket_T_np, blk_plan)


def blocked_overlap_T(walker, trial_data=None):
    """<psi_T|SD(C)>, charge-blocked. A drop-in replacement for mps_overlap_T."""
    bra, ca, cb = _bra(walker)
    return (amp_exact_ref(ca, cb) / amp_mps_ref(bra)) * blocked_overlap(
        bra, ket_T_blk, blk_plan)


print("bond dims      walker", [len(q) for q in qn_bra])
print("               trial ", [len(q) for q in qn_ket])
print("charges/bond   walker", [len(set(map(tuple, q.tolist()))) for q in qn_bra])
print("               trial ", [len(set(map(tuple, q.tolist()))) for q in qn_ket])
print("               shared", [len(c) for c in blk_plan["charges"]],
      "  <- the chi=8 trial is what limits this")
print("padded block   bra   ", blk_plan["A"])
print("               ket   ", blk_plan["B"])
_rep = plan_report(blk_plan)
print(f"""
environment entries   dense {_rep['dense']}
                      charge blocks {_rep['exact_blocks']}
                      padded to rectangles {_rep['padded_blocks']}"""
      f"  ({_rep['dense'] / _rep['padded_blocks']:.1f}x fewer than dense)")
print(f"batched transitions per sweep: {_rep['transitions']}")

bond dims      walker [1, 4, 16, 64, 256, 729, 1089, 1089, 1089, 1089, 1089, 576, 256, 64, 16, 4, 1]
               trial  [1, 4, 16, 48, 64, 64, 64, 64, 64, 64, 64, 64, 64, 51, 16, 4, 1]
charges/bond   walker [1, 4, 9, 16, 25, 25, 25, 25, 25, 25, 25, 25, 25, 16, 9, 4, 1]
               trial  [1, 4, 9, 13, 15, 15, 16, 19, 17, 19, 16, 15, 15, 15, 9, 4, 1]
               shared [1, 4, 9, 13, 15, 15, 16, 19, 17, 19, 16, 15, 15, 15, 9, 4, 1]   <- the chi=8 trial is what limits this
padded block   bra    [1, 1, 4, 9, 36, 100, 225, 225, 225, 225, 225, 100, 36, 9, 4, 1, 1]
               ket    [1, 1, 4, 9, 10, 10, 9, 11, 11, 11, 9, 10, 10, 9, 4, 1, 1]

environment entries   dense 471650
                      charge blocks 38939
                      padded to rectangles 244291  (1.9x fewer than dense)
batched transitions per sweep: 606


In [46]:
"""Two knobs, and a gauge fix that survives them.

WALKER COMPRESSION (CHI_WALKER) caps the CHANNEL bond dimension of the walker
conversion. Carried over from `mps_trial_cpmc.ipynb`: truncation is a discrete
decision, so it goes in the plan -- a NumPy dry run records how many states each
charge sector of each split keeps, which keeps shapes static under vmap while the
singular values are still recomputed from every walker. The splits never need an
SVD of the big matrix; following arXiv:2212.09782, reduce with a QR and read the
Schmidt values off the small hermitian R R^T,

    M = Q R,   R R^T = V S^2 V^T,   M_k = (Q V_k) (V_k^T R)

which is the exact rank-k truncation at about a third of an SVD's cost.

Unlike everything else here this one is an APPROXIMATION: it makes the walker MPS
inexact, so the MPS and MSD routes stop agreeing to machine precision.
CHI_WALKER = None (the default) keeps the conversion exact and the agreement at
1e-14. Note chi_channel is at most 16 at L=8 half filling, so CHI_WALKER >= 16 is
no truncation at all.

THE GAUGE HAD TO CHANGE FOR IT TO BE USABLE. Part 3 fixes the scale between the
walker MPS and the true determinant by matching ONE reference amplitude,

    amp_exact_ref(ca, cb) / amp_mps_ref(bra)

which is exact for an exact MPS -- any amplitude would do -- but divides by a
single truncated number, so it amplifies truncation error violently. The robust
alternative uses no amplitude at all: the conversion is orthogonal up to a sign,

    <MPS|SD(q)> = det(U_rot[occ, :]) = +-1      and     |SD(C)> = det(R) |SD(Q)>

so  g = det(R_a) det(R_b) g_a g_b  with g_sigma handed back by `channel_mps_c`.
Measured against the MSD overlap on the perturbed batch:

    CHI_WALKER   discarded   ratio gauge   det gauge
      None        0.0e+00      3.2e-14      3.1e-15
      8           5.5e-04      6.3e+02      2.6e-01
      4           5.7e-01          inf       9.0e-01

The det gauge is better even in the exact case, and ~2400x better at chi=8. The
residual error at chi=8 is much larger than `discarded` because the bond plan is
frozen on the HF reference while the walkers have drifted away from it.
"""

CHI_WALKER = 20            # None = exact; int caps the CHANNEL bond dimension
CUTOFF_WALKER = 0.0          # additionally drop s < CUTOFF * s_max per split

BondPlan = namedtuple("BondPlan", "ks qn chi discarded")


def sector_plan_ks(ql, qr, ks=None):
    """sector_plan, keeping only ks[s] states in charge sector s."""
    nl, nr = len(ql), len(qr)
    rc = (ql[:, None] + np.arange(2)[None, :]).ravel()
    cc = (qr[None, :] - np.arange(2)[:, None]).ravel()
    secs, qm, rcat, ccat = [], [], [], []
    for s, nm in enumerate(sorted(set(rc.tolist()) & set(cc.tolist()))):
        r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
        k = min(len(r), len(c)) if ks is None else int(ks[s])
        if k == 0:
            continue
        secs.append((r, c, k)); qm += [nm] * k
        rcat.append(r); ccat.append(c)
    rcat, ccat = np.concatenate(rcat), np.concatenate(ccat)
    rmap = np.full(2 * nl, len(rcat), int); rmap[rcat] = np.arange(len(rcat))
    cmap = np.full(2 * nr, len(ccat), int); cmap[ccat] = np.arange(len(ccat))
    return secs, np.array(qm, int), rmap, cmap


_SEC_KS_CACHE = {}

def _sectors_ks(ql, qr, ks):
    key = (ql.tobytes(), len(ql), qr.tobytes(), len(qr), ks)
    if key not in _SEC_KS_CACHE:
        _SEC_KS_CACHE[key] = sector_plan_ks(ql, qr, ks)
    return _SEC_KS_CACHE[key]


def split_trunc(T, ql, qr, ks):
    """Compressing split, exact rank-k per sector, via QR + eigh of the small R R^T."""
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl * 2, 2 * Dr)
    secs, qm, rmap, cmap = _sectors_ks(ql, qr, tuple(ks))
    As, Bs = [], []
    for r, c, k in secs:
        q, rr = jnp.linalg.qr(M[np.ix_(r, c)], mode="reduced")
        _, V = jnp.linalg.eigh(rr @ rr.T)
        Vk = V[:, ::-1][:, :k]
        As.append(q @ Vk); Bs.append(Vk.T @ rr)
    A = jax.scipy.linalg.block_diag(*As)
    B = jax.scipy.linalg.block_diag(*Bs)
    A = jnp.concatenate([A, jnp.zeros((1, A.shape[1]), A.dtype)], 0)[rmap]
    B = jnp.concatenate([B, jnp.zeros((B.shape[0], 1), B.dtype)], 1)[:, cmap]
    return A.reshape(Dl, 2, -1), B.reshape(-1, 2, Dr), qm


def plan_bonds(C, plan, chi_max=None, cutoff=0.0):
    """NumPy dry run: freeze how each split spends its bond budget."""
    occ = plan[0]
    ts = [np.eye(2)[int(o)].reshape(1, 2, 1) for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))
    angles, _ = channel_angles(np.asarray(C, float), plan, xp=np)

    ks_all, discarded = [], 0.0
    for p, th in reversed(angles):
        T = _gate_pair(ts[p], ts[p + 1], th, xp=np)
        Dl, _, _, Dr = T.shape
        M = T.reshape(Dl * 2, 2 * Dr)
        secs, _, _, _ = sector_plan_ks(qn[p], qn[p + 2])
        svs, blocks = [], []
        for r, c, kfull in secs:
            u, sv, vt = np.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
            svs.append(sv[:kfull]); blocks.append((u, sv, vt))
        flat = np.concatenate(svs)
        order = np.argsort(-flat)
        sel = order[: len(flat) if chi_max is None else min(int(chi_max), len(flat))]
        if cutoff > 0.0 and len(flat):
            sel = sel[flat[sel] > cutoff * flat[order[0]]]
        keep = np.zeros(len(flat), bool); keep[sel] = True
        discarded += float((flat[~keep] ** 2).sum())
        ks, off = [], 0
        for sv in svs:
            ks.append(int(keep[off:off + len(sv)].sum())); off += len(sv)
        ks_all.append(tuple(ks))

        _, qm, rmap, cmap = sector_plan_ks(qn[p], qn[p + 2], tuple(ks))
        As, Bs = [], []
        for (u, sv, vt), k in zip(blocks, ks):
            if k:
                As.append(u[:, :k]); Bs.append(sv[:k, None] * vt[:k])
        A = scipy.linalg.block_diag(*As); B = scipy.linalg.block_diag(*Bs)
        A = np.concatenate([A, np.zeros((1, A.shape[1]))], 0)[rmap]
        B = np.concatenate([B, np.zeros((B.shape[0], 1))], 1)[:, cmap]
        ts[p] = A.reshape(Dl, 2, -1); ts[p + 1] = B.reshape(-1, 2, Dr)
        qn[p + 1] = qm
    return BondPlan(ks=ks_all, qn=qn, chi=max(len(q) for q in qn),
                    discarded=discarded)


def channel_mps_c(C, plan, bond_plan=None):
    """channel_mps with optional compression. Returns (tensors, labels, gauge sign)."""
    
    one_hot = (jnp.array([[[1.0], [0.0]]]), jnp.array([[[0.0], [1.0]]]))
    occ = plan[0]
    ts = [one_hot[int(o)] for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))
    angles, rows = channel_angles(C, plan)
    for gi, (p, th) in enumerate(reversed(angles)):
        T = _gate_pair(ts[p], ts[p + 1], th)
        if bond_plan is None:
            ts[p], ts[p + 1], qn[p + 1] = split_full(T, qn[p], qn[p + 2])
        else:
            ts[p], ts[p + 1], qn[p + 1] = split_trunc(T, qn[p], qn[p + 2],
                                                      bond_plan.ks[gi])
    return ts, qn, jnp.linalg.det(jnp.stack([rows[i] for i in np.where(occ == 1)[0]]))


# ------------------------------------------------- the walker conversion we use
bp_walk_a = bp_walk_b = None
if CHI_WALKER is not None or CUTOFF_WALKER > 0.0:
    bp_walk_a = plan_bonds(Ca, plan_a, chi_max=CHI_WALKER, cutoff=CUTOFF_WALKER)
    bp_walk_b = plan_bonds(Cb, plan_b, chi_max=CHI_WALKER, cutoff=CUTOFF_WALKER)


def convert_walker(ca, cb):
    """(d=4 tensors, bond labels, gauge) for one walker. No reference amplitude."""
    qa, ra = _qr_t(ca)
    qb, rb = _qr_t(cb)
    ta, qna, ga = channel_mps_c(qa, plan_a, bp_walk_a)
    tb, qnb, gb = channel_mps_c(qb, plan_b, bp_walk_b)
    ts, qn = combine(ta, qna, tb, qnb)
    return ts, qn, ra * rb * ga * gb


# the block plan has to be rebuilt: truncation changes the walker's bond labels
_bra_ref, qn_bra, _ = convert_walker(jnp.asarray(Ca), jnp.asarray(Cb))
blk_plan = block_plan(qn_bra, qn_ket)
ket_T_blk = ket_blocks(ket_T_np, blk_plan)


def blocked_overlap_T(walker, trial_data=None):
    """<psi_T|SD(C)>, charge-blocked, with the det gauge."""
    ca, cb = walker
    bra, _, gc = convert_walker(ca, cb)
    return gc * blocked_overlap(bra, ket_T_blk, blk_plan)

In [47]:
"""Blocking the ENERGY too: what it takes, and why it is off by default.

The overlap could be blocked because both sides carry (n_a, n_b) labels. The
energy needs <phi|H|psi_T>, and `Hket_T = compress(apply_mpo(W, ket_T))` has no
labels. They are lost in two distinct places:

  * `apply_mpo` -- RECOVERABLE. The output bond is the flattened pair (MPO bond
    dw, MPS bond c), so its charge is mpo_q[dw] + mps_q[c], and the Dw=6 Hubbard
    MPO's bond charges follow from how `hubbard_mpo` builds it: states 0
    ("nothing started") and 5 ("done") carry (0,0); state 1 is entered by cr_a so
    it carries (+1,0); state 2 by an_a -> (-1,0); 3 by cr_b -> (0,+1); 4 by an_b
    -> (0,-1). `apply_mpo_qn` below just carries that through, and the assertion
    checks every nonzero obeys q_left + q_phys = q_right.

  * `compress` -- ACTUALLY DESTROYS THEM. It QRs and SVDs the dense reshaped
    matrices with no charge sorting: the returned basis comes back in
    singular-value order, which interleaves charges arbitrarily; a degenerate
    singular value lets the factorisation MIX vectors of different charge; and
    the rank cut `(sv > tol*sv[0]).sum()` is one global count, so it can keep a
    partial sector. A charge-aware compress (sort by charge, factorise per
    sector, truncate per sector) would fix it -- that is the `sector_plan`
    machinery again, on a d=4 chain.

The way out taken here is simpler: SKIP `compress`. It is exact at tol=1e-13
anyway, so the uncompressed H|psi_T> is the same state, just at bond dimension
6*chi = 48 instead of 38. Verified below on random determinant amplitudes.

AND IT STILL DOES NOT PAY. Measured below: the blocked energy is exact to ~1e-14
but SLOWER than the dense one. The H side has 17 shared charges at the middle
bond with very uneven sector sizes, so padding every sector to a rectangle wastes
7378/1782 = 4.1x, which cancels most of the 11x block sparsity, and the extra
transitions cost more XLA ops. Fixing it needs size-bucketed padding (pad within
groups of similar-sized sectors instead of to one global max), which trades flops
for op count -- a bad trade at L=8, where these kernels are launch-bound.

It is also nearly irrelevant: the propagation needs the overlap at every one of
the 2L+2 = 18 field decisions per step against ONE energy evaluation per block.
At 20 prop steps that is 360 against 1, and the fast sweep serves all 360 from
one conversion plus a site loop. So the energy is ~1% of the work either
way. BLOCK_ENERGY is left False; the machinery is here and verified so the
measurement can be repeated rather than re-argued.
"""
BLOCK_ENERGY = False

# entering MPO bond state k has created this much charge
MPO_QN = np.array([[0, 0], [1, 0], [-1, 0], [0, 1], [0, -1], [0, 0]])


def apply_mpo_qn(W, ts, qn):
    """apply_mpo, carrying the bond labels.

    `apply_mpo` flattens the left bond as a*cl + c with the MPO index major, so a
    combined index has charge mpo_q[a] + mps_q[c].
    """
    out, qout = [], []
    for i, (w, A) in enumerate(zip(W, ts)):
        A = np.asarray(A)
        mq_l = MPO_QN[0:1] if i == 0 else MPO_QN
        if i == 0:
            w = w[0:1]
        if i == len(ts) - 1:
            w = w[:, :, :, 5:6]
        T = np.einsum("apqb,cqd->acpbd", w, A)
        dl, cl, p, dr, cr = T.shape
        out.append(T.reshape(dl * cl, p, dr * cr))
        qout.append(np.concatenate([mq_l[a][None, :] + qn[i] for a in range(dl)], 0))
    qout.append(MPO_QN[5][None, :] + qn[len(ts)])
    return out, qout


Hts_qn, Hqn = apply_mpo_qn(hubbard_mpo(L, t, U), ket_T_np, qn_ket)

_bad = 0
for _x, _A in enumerate(Hts_qn):
    for _l in range(4):
        _dq = np.array([_l % 2, _l // 2])
        for _r, _c in np.argwhere(np.abs(_A[:, _l, :]) > 1e-12):
            _bad += not np.array_equal(Hqn[_x][_r] + _dq, Hqn[_x + 1][_c])
print(f"H|psi_T>  uncompressed dims {[t.shape[0] for t in Hts_qn] + [1]}")
print(f"          compressed   dims {[t.shape[0] for t in Hket_T] + [1]}")
print(f"          charge-violating nonzeros: {_bad}   (must be 0)")

_rng2 = np.random.default_rng(1)
_e = 0.0
for _ in range(20):
    _oa = np.zeros(L, int); _oa[_rng2.choice(L, n_up, replace=False)] = 1
    _ob = np.zeros(L, int); _ob[_rng2.choice(L, n_down, replace=False)] = 1
    _e = max(_e, abs(mps_amp(Hts_qn, _oa, _ob)
                     - mps_amp([np.asarray(t) for t in Hket_T], _oa, _ob)))
print(f"          max |amp(uncompressed) - amp(compressed)| = {_e:.2e}  (same state)")

H_plan = block_plan(qn_bra, Hqn)
Hket_blk = ket_blocks(Hts_qn, H_plan)
_repH = plan_report(H_plan)
print(f"\nH-side env entries: dense {_repH['dense']}, blocks {_repH['exact_blocks']},"
      f" padded {_repH['padded_blocks']}"
      f"   -> {_repH['dense']/_repH['exact_blocks']:.1f}x sparsity but"
      f" {_repH['padded_blocks']/_repH['exact_blocks']:.1f}x padding waste")


def blocked_energy_T(walker, ham_data=None, meas_ctx=None, trial_data=None):
    """E_loc with BOTH contractions charge-blocked."""
    ca, cb = walker
    bra, _, _ = convert_walker(ca, cb)
    return (blocked_overlap(bra, Hket_blk, H_plan)
            / blocked_overlap(bra, ket_T_blk, blk_plan))


def dense_energy_T(walker, ham_data=None, meas_ctx=None, trial_data=None):
    """E_loc through the dense contraction against the compressed H|psi_T>."""
    ca, cb = walker
    bra, _, _ = convert_walker(ca, cb)
    return mps_overlap(bra, Hket_T) / mps_overlap(bra, ket_T)

energy_T = blocked_energy_T if BLOCK_ENERGY else dense_energy_T

H|psi_T>  uncompressed dims [1, 24, 96, 288, 384, 384, 384, 384, 384, 384, 384, 384, 384, 306, 96, 24, 1]
          compressed   dims [1, 4, 16, 59, 125, 196, 242, 234, 228, 238, 241, 196, 126, 61, 16, 4, 1]
          charge-violating nonzeros: 0   (must be 0)
          max |amp(uncompressed) - amp(compressed)| = 1.32e-16  (same state)

H-side env entries: dense 1301186, blocks 94163, padded 449448   -> 13.8x sparsity but 4.8x padding waste
